## tl;dr
Direction-conditioned low-band RGB reduces the held-train-camera proxy error by 50.55%, but neither direct base replacement nor source-anchored transfer removes the visible skin patch. Native gate: 0 pass / 6 fail (two variants, three held views). No accepted temporal or fly-through repair.

## Context & Methods
As of 2026-09-06 UTC, diagnostic frame 000973. Forty-nine train cameras fit degrees 0/1/2; thirteen fixed train cameras evaluate the same visible vertices. A pre-recorded camera-balanced gate selects degree 2, then all 62 train cameras refit. No F/J/L RGB or semantic masks enter fitting.

### Key Assumptions
Earlier camera response used all train cameras: this is only a directional-model holdout. The score compares smoothed source bases, not full images. Face metrics use the same study ROI and exclude neck, tube, hand and ear; they do not replace visual review. This repeatedly inspected time is not untouched generalization.

## Data
Retained root: `/mnt/data/lookcloser_dec5_5a3_surface_repair/view_conditioned_surface_base_control`. Run in the repository `.venv`, with `/home/brans/lookcloser_temp/surface_repair_python_deps` on PYTHONPATH. Independent checks are in the hash-bound audit script linked below.

In [1]:
from pathlib import Path
import hashlib, json, os, subprocess, sys
root = Path('/mnt/data/lookcloser_dec5_5a3_surface_repair/view_conditioned_surface_base_control')
audit = json.loads((root/'integrity_audit.json').read_text())
findings = json.loads((root/'findings.json').read_text())
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()
for path, expected in audit['hashes'].items():
    assert digest(path) == expected, path
assert len(audit['hashes']) == audit['hash_count']
print({'verified_hashes':audit['hash_count'], 'held_train_cameras':audit['held_train_cameras']})

{'verified_hashes': 230, 'held_train_cameras': 13}


## Results
### 1. Recompute the held-camera scores and exact render checks
The read-only audit independently evaluates retained polynomial coefficients against source bases with NumPy, verifies all six original PNG replays and source maps, and recomputes face PSNR from the GT-only polygon. It uses `--check-only`, so retained experiment artifacts are not rewritten.

In [2]:
script = Path('/mnt/data/lookcloser_dec5_5a3_surface_repair/tools/audit_view_conditioned_surface_base.py')
assert digest(script) == audit['hashes'][str(script)]
result = subprocess.run([sys.executable, str(script), '--check-only'], check=True, capture_output=True, text=True)
checks = json.loads(result.stdout)
assert checks['independent_holdout_max_discrepancy'] < 1e-7
assert checks['visual_pass'] == 0 and checks['visual_fail'] == 6
print({'held_medians_constant_linear_quadratic':checks['held_medians'],
       'max_independent_discrepancy':checks['independent_holdout_max_discrepancy']})

{'held_medians_constant_linear_quadratic': [0.020638318495764325, 0.0119866137958587, 0.010205796442559775], 'max_independent_discrepancy': 2.373251269116583e-09}


### 2. Keep face quality separate from color interpolation
Same fixed face ROI, identical GT hash, no candidate surface mask. SSIM/LPIPS are paired retained scorer outputs; exact-pixel PSNR was independently recomputed above.

In [3]:
checks['metrics']

[{'variant': 'original',
  'face_psnr': 28.724285125732422,
  'face_ssim': 0.889098048210144,
  'face_lpips': 0.04798922687768936},
 {'variant': 'direct',
  'face_psnr': 28.110578536987305,
  'face_ssim': 0.8876739144325256,
  'face_lpips': 0.04896348714828491},
 {'variant': 'anchored',
  'face_psnr': 28.22968292236328,
  'face_ssim': 0.8884477019309998,
  'face_lpips': 0.048383645713329315}]

## Takeaways
The direct model gives face 28.110579 / .887674 / .048963; source anchoring gives 28.229683 / .888448 / .048384 versus original 28.724285 / .889098 / .047989. Lower LPIPS is better. Source anchoring fixes the exact source-view identity property, not the visible neck patch or wrong tube/hair contours. Native paired images and verdicts are linked in the Markdown report. Original campaign and defaults are unchanged; no temporal/fly-through acceptance is claimed.